In [1]:
pip install pandas nltk pyodbc sqlalchemy


   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   -------------------- ------------------- 0.8/1.6 MB 5.6 MB/s eta 0:00:01
   ---------------------------------------- 1.6/1.6 MB 5.9 MB/s  0:00:00

   ---------------------------------------- 0/6 [tqdm]
   ---------------------------------------- 0/6 [tqdm]
   ---------------------------------------- 0/6 [tqdm]
   ---------------------------------------- 0/6 [tqdm]
   ---------------------------------------- 0/6 [tqdm]
   ------ --------------------------------- 1/6 [regex]
   ------ --------------------------------- 1/6 [regex]
   ------------- -------------------------- 2/6 [pyodbc]
   -------------------- ------------------- 3/6 [joblib]
   -------------------- ------------------- 3/6 [joblib]
   -------------------- ------------------- 3/6 [joblib]
   -------------------- ------------------- 3/6 [joblib]
   -------------------- ------------------- 3/6 [joblib]
   -------------------- ------------------- 3/6 [jo

In [2]:
import pandas as pd
import pyodbc
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

In [3]:
# download the VADER lexicon for sentiment analysis if not already present

nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Anwesha\AppData\Roaming\nltk_data...


True

In [9]:
# define a function to fetch data from a SQL database using SQL query

def fetch_data_from_sql():
    
    # define the connection string with parameters for the database connection
    conn_str = (
        "Driver={SQL Server};"
        "Server=LAPTOP-QG103FL2\SQLEXPRESS;"
        "Database=PortfolioProject_MarketingAnalytics;"
        "Trusted_Connection=yes;" # use windows authentication for connection
    )
    
    # establish the connection to the database
    conn = pyodbc.connect(conn_str)
    
    # define the sql query to fetch the customer review data
    query = "SELECT ReviewID, ProductID, ReviewDate, Rating, ReviewText from customer_reviews"
    
    # execute the query and fetch the data into a dataframe
    df = pd.read_sql(query, conn)
    
    # close the connection to free up the resources
    conn.close()
    
    # return fetched data as a dataframe
    return df

In [10]:
# fetch customer reviews data from the SQL database

customer_reviews_df = fetch_data_from_sql()

C:\Users\Anwesha\AppData\Local\Temp\ipykernel_11028\394646015.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [11]:
print(customer_reviews_df.head())

   ReviewID  ProductID  ReviewDate  Rating  \
0         1         18  2023-12-23       3   
1         2         19  2024-12-25       5   
2         3         13  2025-01-26       4   
3         4         15  2025-04-21       3   
4         5          2  2023-07-16       3   

                                 ReviewText  
0   Average  experience,  nothing  special.  
1            The  quality  is    top-notch.  
2   Five  stars  for  the  quick  delivery.  
3  Good  quality,  but  could  be  cheaper.  
4   Average  experience,  nothing  special.  


In [12]:
# intialises the VADER sentiment intensity analyzer for analyzing the sentiment of text data

sia = SentimentIntensityAnalyzer()

In [13]:
# define a function to calculate sentiment scores using VADER

def calculate_sentiment(review):
    
    # get sentiment scores for the review text
    sentiment = sia.polarity_scores(review)
    
    # return the compound score, which is a normalised score between -1 (most negative) and 1 (most positive)
    return sentiment['compound']
    

In [17]:
# define a function to categorise the sentiment using both the sentiment score and the review rating

def categorise_sentiment(score, rating):

    # Use both the text sentiment score and the numerical rating to determine sentiment category
    if score > 0.05:         # positive sentiment score
        if rating >= 4:      # high rating and positive sentiment
            return 'Positive'
        elif rating == 3:    # neutral rating but positive sentiment
            return 'Mixed Positive'
        else:                # low rating but positive sentiment   
            return 'Mixed Negative'
            
    elif score < -0.05:      # negative sentiment score
        if rating <= 2:      # low rating and negative sentiment
            return 'Negative'
        elif rating == 3:    # neutral rating but negative sentiment
            return 'Mixed Negative'
        else:                # positive rating but negative sentiment
            return 'Mixed Positive'
            
    else:                    # neutral sentiment score
        if rating >= 4:      # high rating but neutral sentiment
            return 'Positive'
        elif rating <= 2:    # low rating and neutral sentiment 
            return 'Negative'
        else:                # neutral rating and neutral sentiment
            return 'Neutral'
        
        
        

In [19]:
# define a function to bucket sentiment score into text ranges

def sentiment_bucket(score):
    if score >= 0.5:         # strongly positive sentiment
        return '0.5 to 1.0'
    elif 0.0 <= score < 0.5:  # mildly positive sentiment
        return '0.0 to 0.49'
    elif -0.5 <= score < 0.0:  # mildly negative sentiment
        return '-0.49 to 0.0'
    else:                     # strongly negative sentiment
        return '-1.0 to -0.5'
        

In [20]:
# apply sentiment analysis to calculate sentiment scores for each review

customer_reviews_df['SentimentScore'] = customer_reviews_df['ReviewText'].apply(calculate_sentiment)

In [21]:
# apply sentiment categorisation using both text and rating

customer_reviews_df['SentimentCategory'] = customer_reviews_df.apply(
    lambda row: categorise_sentiment(row['SentimentScore'], row['Rating']), axis=1)

In [22]:
# apply sentiment bucketing to categorise score into defined ranges

customer_reviews_df['SentimentBucket'] = customer_reviews_df['SentimentScore'].apply(sentiment_bucket)

In [23]:
# Display the first few rows of the DataFrame with sentiment scores, categories, and buckets

print(customer_reviews_df.head())

   ReviewID  ProductID  ReviewDate  Rating  \
0         1         18  2023-12-23       3   
1         2         19  2024-12-25       5   
2         3         13  2025-01-26       4   
3         4         15  2025-04-21       3   
4         5          2  2023-07-16       3   

                                 ReviewText  SentimentScore SentimentCategory  \
0   Average  experience,  nothing  special.         -0.3089    Mixed Negative   
1            The  quality  is    top-notch.          0.0000          Positive   
2   Five  stars  for  the  quick  delivery.          0.0000          Positive   
3  Good  quality,  but  could  be  cheaper.          0.2382    Mixed Positive   
4   Average  experience,  nothing  special.         -0.3089    Mixed Negative   

  SentimentBucket  
0    -0.49 to 0.0  
1     0.0 to 0.49  
2     0.0 to 0.49  
3     0.0 to 0.49  
4    -0.49 to 0.0  


In [24]:
# Save the DataFrame with sentiment scores, categories, and buckets to a new CSV file

customer_reviews_df.to_csv('fact_customer_reviews_with_sentiment.csv', index=False)